In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=10000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([8000, 10]) torch.Size([8000, 1])
torch.Size([2000, 10]) torch.Size([2000, 1])


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=2e-2)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.08212599903345108
epoch:  1, loss: 0.05568627640604973
epoch:  2, loss: 0.042899180203676224
epoch:  3, loss: 0.03670968487858772
epoch:  4, loss: 0.03371572867035866
epoch:  5, loss: 0.032269321382045746
epoch:  6, loss: 0.03157159686088562
epoch:  7, loss: 0.031235573813319206
epoch:  8, loss: 0.031074056401848793
epoch:  9, loss: 0.030996588990092278
epoch:  10, loss: 0.03095952235162258
epoch:  11, loss: 0.03094181790947914
epoch:  12, loss: 0.03093336708843708
epoch:  13, loss: 0.030929308384656906
epoch:  14, loss: 0.030927326530218124
epoch:  15, loss: 0.03092632070183754
epoch:  16, loss: 0.03092576563358307
epoch:  17, loss: 0.030925417318940163
epoch:  18, loss: 0.03092515841126442
epoch:  19, loss: 0.030924933031201363
epoch:  20, loss: 0.030924715101718903
epoch:  21, loss: 0.03092450089752674
epoch:  22, loss: 0.03092428855597973
epoch:  23, loss: 0.030924074351787567
epoch:  24, loss: 0.030923860147595406
epoch:  25, loss: 0.030923645943403244
epoch:  2

ValueError: too many values to unpack (expected 2)

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -590.7427292144325
Test metrics:  R2 = -607.1822580886663


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=1e-2)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5636163353919983
epoch:  1, loss: 0.1359165608882904
epoch:  2, loss: 0.0604143925011158
epoch:  3, loss: 0.039643969386816025
epoch:  4, loss: 0.028337236493825912
epoch:  5, loss: 0.02702736295759678
epoch:  6, loss: 0.023730875924229622
epoch:  7, loss: 0.020253458991646767
epoch:  8, loss: 0.017084747552871704
epoch:  9, loss: 0.01325621735304594
epoch:  10, loss: 0.01017664186656475
epoch:  11, loss: 0.008745120838284492
epoch:  12, loss: 0.007985771633684635
epoch:  13, loss: 0.007822364568710327
epoch:  14, loss: 0.007695029955357313
epoch:  15, loss: 0.007610576692968607
epoch:  16, loss: 0.0075103724375367165
epoch:  17, loss: 0.007477599661797285
epoch:  18, loss: 0.007396260742098093
epoch:  19, loss: 0.007333222310990095
epoch:  20, loss: 0.007240767125040293
epoch:  21, loss: 0.007128403056412935
epoch:  22, loss: 0.0069719343446195126
epoch:  23, loss: 0.006772053427994251
epoch:  24, loss: 0.006538025103509426
epoch:  25, loss: 0.006302177906036377
epo

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9943573185001566
Test metrics:  R2 = 0.9941366436402294
